# Väderanalys och modellförsök

Syftet är att undersöka vädervariablernas samband med uthyrningen
och utvärdera en första modell med enbart väderdata.

## Datakälla

[Seoul Bike Sharing Demand – UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

In [ ]:
df = pd.read_csv(
    "../data/SeoulBikeData.csv",
    encoding="cp1252"
)

df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

**Inga saknade värden**

---

In [ ]:
df.duplicated().sum()

**Inga dubletter**

---

In [ ]:
df["Rented Bike Count"].describe()

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(data=df, x="Rented Bike Count", bins=30)

plt.title("Fördelning av antal uthyrda cyklar per timme")
plt.xlabel("Antal uthyrda cyklar")
plt.ylabel("Antal timmar")
plt.show()

**Diagrammet visar att timmar med låg uthyrning är vanligare än timmar med mycket hög uthyrning.**

---

In [ ]:
df.groupby("Functioning Day")["Rented Bike Count"].agg(
    ["count", "min", "mean", "max"]
)

In [ ]:
df_open = df[df["Functioning Day"] == "Yes"].copy()

Vi fokuserar på uthyrning under timmar då systemet är i drift. Därför exkluderar vi 295 stängda timmar med noll uthyrningar. Dessa nollor visar att uthyrning inte var möjlig, snarare än att efterfrågan saknades.

---

## 1. Temperaturdiagram

In [ ]:
plt.figure(figsize=(10, 5))

sns.scatterplot(
    data=df_open,
    x="Temperature(°C)",
    y="Rented Bike Count",
    alpha=0.4,
    s=18,
)

plt.title("Samband mellan temperatur och cykeluthyrning")
plt.xlabel("Temperatur (°C)")
plt.ylabel("Antal uthyrningar per timme")
plt.show()

**Temperatur har ett positivt samband med cykeluthyrningen, men spridningen är stor vid samma temperatur. Det motiverar att testa temperatur som indata och undersöka om en kombination av väder och tidsinformation förbättrar förutsägelserna.**

---

## 2. Boxplot för regn

In [ ]:
df_open["Rain category"] = (
    df_open["Rainfall(mm)"] > 0
).map({True: "Regn", False: "Inget regn"})

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_open,
    x="Rain category",
    y="Rented Bike Count",
    order=["Inget regn", "Regn"],
    color="steelblue",
)

plt.title("Cykeluthyrning under timmar med och utan regn")
plt.xlabel("Registrerad nederbörd")
plt.ylabel("Antal uthyrningar per timme")
plt.show()

**Uthyrningen har betydligt lägre median under regniga timmar än under timmar utan regn. Det motiverar att testa nederbörd som indata till modellen. Grupperna innehåller 516 respektive 7 949 timmar, och jämförelsen tar inte hänsyn till klockslag, temperatur eller årstid.**

In [ ]:
df_open["Rain category"].value_counts()

---

## 3. Korrelationskarta

In [ ]:
weather_columns = [
    "Temperature(°C)",
    "Humidity(%)",
    "Wind speed (m/s)",
    "Visibility (10m)",
    "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)",
    "Rainfall(mm)",
    "Snowfall (cm)",
    "Rented Bike Count"
]

weather_corr = df_open[weather_columns].corr()

plt.figure(figsize=(9, 7))

sns.heatmap(
    weather_corr,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

plt.title("Korrelation mellan väder och cykeluthyrning under öppna timmar")
plt.tight_layout()
plt.show()

**Temperatur har starkast positivt linjärt samband med uthyrningen bland vädervariablerna. Temperatur och daggpunkt samvarierar starkt, vilket motiverar att undersöka om båda behövs i modellen. Regn behålls som kandidat trots låg linjär korrelation, eftersom boxploten visar tydligt lägre medianuthyrning under regniga timmar.**

---

## 4. Modellförsök med väderdata

Vi testar hur väl vädervariabler kan förutsäga uthyrningen under
öppna timmar. Försöket är preliminärt.

In [ ]:
weather_features = [
    column for column in weather_columns
    if column != "Rented Bike Count"
]

X_weather = df_open[weather_features]
y_weather = df_open["Rented Bike Count"]

print("Indata:", X_weather.shape)
print("Målvariabel:", y_weather.shape)

In [ ]:
X_weather_train, X_weather_test, y_weather_train, y_weather_test = (
    train_test_split(
        X_weather,
        y_weather,
        test_size=0.2,
        random_state=42
    )
)

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_weather_train, y_weather_train)

baseline_predictions = baseline.predict(X_weather_test)

baseline_mae = mean_absolute_error(
    y_weather_test,
    baseline_predictions
)

print(f"Baslinjens MAE: {baseline_mae:.1f} uthyrningar per timme")

In [ ]:
weather_model = LinearRegression()
weather_model.fit(X_weather_train, y_weather_train)

weather_predictions = weather_model.predict(X_weather_test)

weather_mae = mean_absolute_error(
    y_weather_test,
    weather_predictions
)

print(f"Baslinjens MAE: {baseline_mae:.1f}")
print(f"Vädermodellens MAE: {weather_mae:.1f}")

**Linjär regression med åtta vädervariabler gav MAE 348,9, jämfört med baslinjens 509,7. Väderinformationen minskade därmed det genomsnittliga absoluta felet med cirka 32 % på denna testuppdelning.**

---

In [ ]:
weather_rmse = root_mean_squared_error(
    y_weather_test, weather_predictions
)

weather_r2 = r2_score(
    y_weather_test, weather_predictions
)

print(f"MAE: {weather_mae:.1f}")
print(f"RMSE: {weather_rmse:.1f}")
print(f"R²: {weather_r2:.3f}")

Vädermodellen fick **MAE 348,9**, **RMSE 470,6** och **R² 0,436**. Modellen fångar en del av variationen i uthyrningen, men gör fortfarande betydande fel. RMSE är högre än MAE eftersom större missar får större tyngd. Resultaten gäller den nuvarande slumpmässiga testuppdelningen.

---

In [ ]:
plt.figure(figsize=(7, 7))

sns.scatterplot(
    x=y_weather_test,
    y=weather_predictions,
    alpha=0.3,
    s=18
)

limits = [
    min(y_weather_test.min(), weather_predictions.min()),
    max(y_weather_test.max(), weather_predictions.max())
]

plt.plot(limits, limits, "--", color="red", label="Perfekt förutsägelse")
plt.xlim(limits)
plt.ylim(limits)
plt.xlabel("Verkligt antal uthyrningar")
plt.ylabel("Förutsagt antal uthyrningar")
plt.title("Vädermodellens förutsägelser på testdatan")
plt.legend()
plt.show()
